In [ ]:
import sys
sys.path.append('..')

In [ ]:
from wavelet_transform.utils.bubble_data_manager import resample
import numpy as np
import plotly.graph_objects as go
from wavelet_transform.utils.statistical_features import kurtosis
from wavelet_transform.utils.bubble_data_manager import generate_data, _get_params_from_kwargs
from wavelet_transform.utils.extensions import psd, to_dB

fs = int(1e9)
t = np.arange(0, 100e-6, 1 / fs)
# broadband-ish test signal
test_params = _get_params_from_kwargs(1e-6, 1e5, -100e3, 10)
x = generate_data(500e-6, *test_params)  # bd data for testing

# x = x + np.random.normal(0, 1, len(t)) * 0.1 * np.std(x)

# x = np.random.normal(0, 1, len(t))

# Downsample to 60 MHz and resample back
fs2 = 60e6
y, t2 = resample(x, fs2, t)

p1, f1 = psd(x, fs)
p2, f2 = psd(y, fs2)

print("Original signal length:", len(x))
print("Resampled signal length:", len(y))

energy_rel_diff = (np.mean(x**2) - np.mean(y**2)) / np.mean(x**2)
print(f"Energy relative difference after resampling: {energy_rel_diff:.4f}")

kurtosis_rel_diff = (kurtosis(x) - kurtosis(y))
print(f"Kurtosis difference after resampling: {kurtosis_rel_diff:.4f}")

psd_rel_diff = np.sum(np.abs(p1[:len(p2)] - p2[:len(p1)])) / np.sum(p1)
print(f"PSD relative difference after resampling: {psd_rel_diff:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=t * 1e6, y=x, name="Original Signal"))
fig.add_trace(go.Scatter(x=t2 * 1e6, y=y, name="Resampled Signal", line=dict(dash="solid")))
fig.update_layout(
    title="Signal Before and After Resampling",
    xaxis_title="Time (µs)",
    yaxis_title="Amplitude",
)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=f1, y=to_dB(p1), name="Original Signal PSD"))
fig.add_trace(go.Scatter(x=f2, y=to_dB(p2), name="Resampled Signal PSD", line=dict(dash="solid")))
fig.update_layout(
    title="Power Spectral Density Before and After Resampling",
    xaxis_title="Frequency (Hz)",
    yaxis_title="Power/Frequency (dB/Hz)",
)
fig.show()